In [1]:
!sudo apt-get install -y fonts-nanum* | tail -n 1
!sudo fc-cache -fv
!rm -rf ~/.cache/matplotlib

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 4.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...
/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 3 dirs
/usr/share/fonts/truetype/humor-sans: caching, new cache contents: 1 fonts, 0 dirs
/usr/share/fonts/truetype/liberation: caching, new cache contents: 16 fonts, 0 dirs
/usr/share/fonts/truetype/nanum: caching, new cache contents: 39 fonts, 0 dirs
/usr/local/share/fonts: caching, new cache contents: 0 fonts, 0 dirs
/root/.local/share/fonts: skipping, no suc

In [2]:
# 필요 라이브러리 설치

!pip install torchviz | tail -n 1
!pip install torchinfo | tail -n 1

세션 다시 시작

In [1]:
# 라이브러리 임포트

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# 폰트 관련 용도
import matplotlib.font_manager as fm

# 나눔 고딕 폰트의 경로 명시
path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_name = fm.FontProperties(fname=path, size=10).get_name()

In [2]:
"""
Part 1: 회귀 손실함수 (MSE, MAE, Huber)
Part 2: 분류 손실함수 (BCE, CrossEntropy)
Part 3: 라벨 스무딩
Part 4: 클래스 불균형 대응
Part 5: 손실 곡면

필수 라이브러리:
pip install torch numpy matplotlib seaborn scikit-learn
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error

# 재현성
torch.manual_seed(42)
np.random.seed(42)


In [6]:
# 회귀 손실함수

y_true = torch.tensor([10.0, 20.0, 30.0,40.0])
y_pred = torch.tensor([12.0, 19.0, 35.0,38.0])

mse = nn.MSELoss()(y_true, y_pred)
mae = nn.L1Loss()(y_true, y_pred)
huber = nn.HuberLoss()(y_true, y_pred)

print(mse.item(), mae.item(), huber.item())

8.5 2.5 2.0


In [10]:
# 이상치 포함시

y_pred_outlier = torch.tensor([12.0, 19.0, 100.0, 38.0])

mse = nn.MSELoss()(y_true, y_pred_outlier)
mae = nn.L1Loss()(y_true, y_pred_outlier)
huber = nn.HuberLoss()(y_true, y_pred_outlier)

print(mse.item(), mae.item(), huber.item())
# mse는 이상치에 너무 민감해 (144배 차이남)
# mse: 큰 오차에 민감, mae: 모든 오차 동등, huber: mse와 mae 절충

1227.25 18.75 18.25


In [12]:
# 분류 손실함수

# BCE 예시
y_true_bin = torch.tensor([1.0, 0.0, 1.0, 0.0])
y_pred_conf = torch.tensor([0.9, 0.1, 0.85, 0.15]) # 확신있는 예측
y_pred_unce = torch.tensor([0.6, 0.4, 0.55, 0.45])  # 불확실한 예측

bce = nn.BCELoss()
loss_conf = bce(y_pred_conf, y_true_bin)
loss_unce = bce(y_pred_unce, y_true_bin)

print(loss_conf.item(), loss_unce.item())
# 확신있을 수록 손실 감소

0.133939728140831 0.5543313026428223


In [ ]:
# BCEWithLogitsLoss vs BCE
# 뭐가 좋아요? BCEWithLogitsLoss
# 왜요? 수치적 안정성, 출력층에 sigmoid 제거

In [14]:
# 라벨 스무딩 비교


# 분류 데이터
X, y = make_classification(n_samples=500, n_features=20,
                          n_classes=3, n_informative=15, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)

# 간단한 모델
class SimpleNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, n_classes)
        )
    def forward(self, x):
        return self.net(x)

# 일반 vs 라벨 스무딩
print("\n라벨 스무딩 비교:")

# 일반
model_no_smooth = SimpleNet(3)
criterion_no_smooth = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_no_smooth.parameters(), lr=0.01)

for _ in range(30):
    optimizer.zero_grad()
    loss = criterion_no_smooth(model_no_smooth(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

# 라벨 스무딩 (PyTorch 1.10+)
try:
    model_smooth = SimpleNet(3)
    criterion_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model_smooth.parameters(), lr=0.01)

    for _ in range(30):
        optimizer.zero_grad()
        loss = criterion_smooth(model_smooth(X_train_t), y_train_t)
        loss.backward()
        optimizer.step()

    print("  라벨 스무딩 (alpha=0.1) 적용 가능")
    print("  효과: 과신 방지, 일반화 향상")
except:
    print("  PyTorch 1.10 미만: label_smoothing 파라미터 없음")


라벨 스무딩 비교:
  라벨 스무딩 (alpha=0.1) 적용 가능
  효과: 과신 방지, 일반화 향상


In [15]:
# 클래스 불균형

In [16]:
# 불균형 데이터 (95:5)
X_imb, y_imb = make_classification(
    n_samples=500, n_features=20,
    weights=[0.95, 0.05], random_state=42
)

print(f"\n불균형 데이터:")
print(f"  클래스 0: {np.sum(y_imb==0)}개 ({np.sum(y_imb==0)/len(y_imb)*100:.0f}%)")
print(f"  클래스 1: {np.sum(y_imb==1)}개 ({np.sum(y_imb==1)/len(y_imb)*100:.0f}%)")


불균형 데이터:
  클래스 0: 473개 (95%)
  클래스 1: 27개 (5%)


In [17]:
# 데이터 분할
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42
)

In [18]:
# 정규화
scaler_i = StandardScaler()
X_train_i = scaler_i.fit_transform(X_train_i)
X_test_i = scaler_i.transform(X_test_i)

In [19]:
# 텐서로 변환
X_train_i_t = torch.FloatTensor(X_train_i)
y_train_i_t = torch.FloatTensor(y_train_i).unsqueeze(1)
X_test_i_t = torch.FloatTensor(X_test_i)

In [20]:
# 가중치 계산
n0 = np.sum(y_train_i == 0)
n1 = np.sum(y_train_i == 1)
weight = n0 / n1

print(f'class weight: {weight:.2f}')


class weight: 15.67


In [21]:
# Weighted BCE
class BinClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

In [22]:
model_weighted = BinClassifier()
nn.BCEWithLogitsLoss(pos_weight=torch.tensor([weight]))
# pos_weight : 양성 클래스(1)에 대한 가중치
# y=1 : Wpos * (-log p_hat) : 소수 클래스 틀리면 큰 패널티 부여
# y=0 : -log (1-p_hat) : 다수 클래스 틀리면 보통 패널티 부여

BCEWithLogitsLoss()